In [2]:
import pandas as pd
import duckdb
import os
import glob

In [3]:
#read files into dictionary
dfs = {}
for file in glob.glob(os.path.join("clean_data", "*.parquet")): #search for all parquet files in clean_data, and iterate
    name = os.path.basename(file).replace(".parquet", "")
    dfs[name] = pd.read_parquet(file)

ANALYSIS 1: HAVE SPECIALISED MOLDS REPLACED ASSEMBLIES OF SMALLER PARTS TO CREATE THE SAME SHAPE? 
...OR VICE VERSA?

THE 'PART_RELATIONSHIPS' TABLE SHOWS 'PARENT' PARTS THAT HAVE RELATED 'CHILD' PARTS.
THESE 'CHILD' PARTS ARE EITHER SEGMENTS THAT COMBINE TO MAKE UP THE PARENT, 
OR THEY ARE A COMINBATION OF MULTIPLE child PARTS.
EITHER WAY, THE child(S) ARE THE PART(S) THAT WERE RELEASED FIRST.

TABLES NEEDED (6 out of 12):

    part_relationships - core table to identify child-child connections
    parts - to get the part name, for human readability
    part_categories - allow for more granular analysis by part category
    inventory_parts, inventories, sets - the 'year' column in the 'sets' table is needed to get the timeline of parent and child part usage


---note: the 'inventory_sets' table is not needed as it just records the quantity of a given part used per set, which is not needed here

In [5]:
part_relationships = dfs["part_relationships"]
parts = dfs["parts"]
part_categories = dfs["part_categories"]
inventory_parts = dfs["inventory_parts"]
inventories = dfs["inventories"]
inventory_sets = dfs["inventory_sets"]
sets = dfs["sets"]

In [6]:
#only look at the parts that have a parent-child relatiosnhip, not eg. a mold update or print
part_relationships = duckdb.sql("SELECT * FROM part_relationships WHERE rel_type = 'R'").df()

In [7]:
#joined table is symmetrical in structure, where parent part info starts from the middle and continues through the left columns,
#and child part indo starts from the middle and continues through the right columnd
joined = duckdb.sql("""
                    
                    WITH parts_sets AS(
                        SELECT
                            p.part_num,
                            s.set_num,
                            s.year,
                            ivs.quantity
                        FROM parts p JOIN inventory_parts ip ON p.part_num = ip.part_num
                        JOIN inventories i ON ip.inventory_id = i.id
                        JOIN inventory_sets ivs ON i.set_num = ivs.set_num
                        JOIN sets s ON ivs.set_num = s.set_num
                    )

                    SELECT DISTINCT ON(pr.parent_part_num, pr.child_part_num)
                        
                        pc.name AS category,                         --only needs to be shows once, as parent and child likely to have same category 
                    
                        --parent_ps.quantity AS parent_quantity,
                        MIN(parent_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS min_parent_year,
                        MAX(parent_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS max_parent_year,
                        --parent_ps.set_num AS parent_set_sum,
                        pr.parent_part_num,
                        parent_p.name AS parent_part_name,
                    
                        child_p.name AS child_part_name,
                        pr.child_part_num,
                        --child_ps.set_num AS child_set_num,
                        MIN(child_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS min_child_year,
                        MAX(child_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS max_child_year
                        --child_ps.quantity AS child_quantity
                    
                    FROM part_relationships pr
                    JOIN parts child_p ON pr.child_part_num = child_p.part_num
                    JOIN parts parent_p ON pr.parent_part_num = parent_p.part_num
                    JOIN part_categories pc ON child_p.part_cat_id = pc.id

                    JOIN parts_sets parent_ps ON parent_p.part_num = parent_ps.part_num
                    JOIN parts_sets child_ps ON child_p.part_num = child_ps.part_num

                    WHERE pr.rel_type = 'R' --only ~8 percent of all parts have this relationship

                    ORDER BY pc.name, pr.parent_part_num, parent_ps.year ASC, child_ps.year ASC --ordering by the years ascending forces the DISTINCT ON to select the minimum
                    


                    
                    """).df() #takes 3.2 - 3.3 seconds to executed

In [8]:
joined


,category,min_parent_year,max_parent_year,parent_part_num,parent_part_name,child_part_name,child_part_num,min_child_year,max_child_year
0,Animal / Creature Body Parts,2014,2014,11777pr0001,"Animal Body Part, Bird, Eagle Wing - Left with...","Animal Body Part, Bird, Eagle Body with Beak, ...",11435pr0002,2014,2014
1,Animal / Creature Body Parts,2014,2014,11778pr0001,"Animal Body Part, Bird, Eagle Wing - Right wit...","Animal Body Part, Bird, Eagle Body with Beak, ...",11435pr0002,2014,2014
2,Animal / Creature Body Parts,2014,2014,11778pr0001,"Animal Body Part, Bird, Eagle Wing - Right wit...","Animal Body Part, Bird, Eagle Wing - Left with...",11777pr0001,2014,2014
3,Animal / Creature Body Parts,2014,2014,16875pr0001,"Creature Body Part, Dewback Body, Claws and Sh...","Creature Body Part, Dewback Lower Jaw with Tee...",16873pr0001,2014,2014
4,Animal / Creature Body Parts,2014,2014,18153pr0001,"Creature Body Part, Dragon Head Upper Jaw with...","Creature Body Part, Dragon Neck, S-Curve with ...",18234,2014,2014
...,...,...,...,...,...,...,...,...,...
973,Windscreens and Fuselage,1994,2003,4625,Hinge Tile 1 x 4,Windscreen 1 x 4 x 1 1/3 with Bottom Hinge,30161,2002,2003
974,Windscreens and Fuselage,2006,2023,54096,"Slope, Curved 8 x 8 x 2 Double with Cutout",Door 2 x 4 x 6 Curved Aircraft,54097,2006,2023
975,Windscreens and Fuselage,2022,2022,71073,Axle for Spinner,Dome for Ninjago Spinner,69783,2022,2022
976,Windscreens and Fuselage,2012,2021,87613,Aircraft Fuselage Curved Forward 6 x 10 Top,Glass for Aircraft Fuselage Curved Forward 6 x...,87612,2012,2021


- DISCOVERY: ACROSS THE BOARD, THE DATABSSE DOES NOT CONSIDER TWO PARTS THAT STRUCTURALLY COMBINE TO MAKE THE THIRD AS A 'PARENT-CHILD' RELATIONSHIP 
- IN FACT, THERE IS NO RELATIOSNHIP WHATSOEVER. EG. BETWEEN PART 51739, AND 24299 WITH 24307. THIS IS LIKELY BECAUSE THE THE CONNECTIONS UNDERNEATH THE PIECES DIFFER. 
- SO, I WILL HAVE TO USE MY OWN KNOWLEDGE OF PARTS TO PIECE TOGETHER MY ANALYSIS

In [45]:
mydf = duckdb.sql("SELECT * FROM part_relationships WHERE parent_part_num = '51739' OR child_part_num = '51739'")
mydf

┌──────────┬────────────────┬─────────────────┐
│ rel_type │ child_part_num │ parent_part_num │
│ varchar  │    varchar     │     varchar     │
└──────────┴────────────────┴─────────────────┘
                    0 rows                   

In [ ]:
#SOME SPECIFIC INSTANCES OF PARENT-CHILD PART RELATIONSHIPS ARE PRESENT AFTER SOME MANUAL INSPECTIONS
#EXTRACT THE MAJORITY BY EXCLUDING WHERE MINIMUM PARENT YEAR = MINIMUM CHILD YEAR...
#...BECAUSE PARTS THAT WERE CREATED IN THE SAME YEAR ARE LIKELY TO BE PART MIRRORINGS, RATHER THAN 'TRUE' PARENT AND CHILD PARTS

joined = duckdb.sql("SELECT * FROM joined WHERE min_parent_year != min_child_year").df()
joined

,category,min_parent_year,max_parent_year,parent_part_num,parent_part_name,child_part_name,child_part_num,min_child_year,max_child_year
0,Animal / Creature Body Parts,2015,2021,20512pr0001,"Animal Body Part, Shark Head with Rounded Nose...","Animal Body Part, Shark Body with Three Gill S...",14518,2013,2021
1,Animal / Creature Body Parts,1991,2013,2547,"Animal Body Part, Shark Body, without Bottom Tube","Animal Body Part, Shark Head with Rounded Nose...",87587,2013,2013
2,Animal / Creature Body Parts,2003,2005,40373,"Animal / Creature Body Part, Dinosaur Body Qua...","Animal / Creature Body Part, Dinosaur Neck / T...",40375,2001,2005
3,Animal / Creature Body Parts,2003,2005,40374,"Animal / Creature Body Part, Dinosaur Body Qua...","Animal / Creature Body Part, Dinosaur Neck / T...",40375,2001,2005
4,Animal / Creature Body Parts,2001,2005,40375,"Animal / Creature Body Part, Dinosaur Neck / T...","Animal Body Part, Elephant Head with Pin",43890c01,2003,2003
...,...,...,...,...,...,...,...,...,...
480,Windscreens and Fuselage,1983,2006,4315,Hinge Vehicle Roof Holder 1 x 4,Windscreen 6 x 4 x 2 Canopy,4474,1985,1991
481,Windscreens and Fuselage,1983,2006,4315,Hinge Vehicle Roof Holder 1 x 4,Windscreen 6 x 6 x 3 Canopy Half Sphere with H...,30083,2001,2003
482,Windscreens and Fuselage,1994,2003,4625,Hinge Tile 1 x 4,Windscreen 6 x 4 x 2 Canopy,4474,1985,1991
483,Windscreens and Fuselage,1994,2003,4625,Hinge Tile 1 x 4,Windscreen 6 x 6 x 3 Canopy Half Sphere with H...,30083,2001,2003


In [33]:
#for each row of parent and child parts, find the quantity of each part used across all sets, for each year that part was in production
part_qty_per_set = duckdb.sql(
"""
SELECT 
    pc.name AS category,
    p.part_num,
    p.name AS part_name,
    s.set_num,
    s.year,
    ip.quantity
FROM parts p
JOIN part_categories pc ON p.part_cat_id = pc.id
JOIN inventory_parts ip ON p.part_num = ip.part_num
JOIN inventories i ON ip.inventory_id = i.id
JOIN sets s ON i.set_num = s.set_num
WHERE p.part_num IN(SELECT parent_part_num FROM joined)
OR p.part_num IN(SELECT child_part_num FROM joined)

ORDER BY pc.name, s.year, s.set_num
"""
).df()

part_qty_per_set

,category,part_num,part_name,set_num,year,quantity
0,Animal / Creature Body Parts,2547,"Animal Body Part, Shark Body, without Bottom Tube",6257-1,1989,1
1,Animal / Creature Body Parts,2547,"Animal Body Part, Shark Body, without Bottom Tube",6270-1,1989,1
2,Animal / Creature Body Parts,2547,"Animal Body Part, Shark Body, without Bottom Tube",6270-1,1989,1
3,Animal / Creature Body Parts,2547,"Animal Body Part, Shark Body, without Bottom Tube",6270-1,1989,1
4,Animal / Creature Body Parts,2547,"Animal Body Part, Shark Body, without Bottom Tube",6270-1,1989,1
...,...,...,...,...,...,...
68763,Windscreens and Fuselage,18907,Aircraft Fuselage Curved Forward 6 x 10 Top wi...,60096-1,2015,1
68764,Windscreens and Fuselage,18908,Glass for Aircraft Fuselage Curved Forward 6 x...,60096-1,2015,1
68765,Windscreens and Fuselage,18907,Aircraft Fuselage Curved Forward 6 x 10 Top wi...,60164-1,2017,1
68766,Windscreens and Fuselage,18908,Glass for Aircraft Fuselage Curved Forward 6 x...,60164-1,2017,1


In [43]:
part_qty_per_year = duckdb.sql(
"""
SELECT 
    part_num,
    year,
    SUM(quantity)::int AS quantity
FROM part_qty_per_set
GROUP BY part_num, year
ORDER BY part_num, year
"""   
).df()

part_qty_per_year

,part_num,year,quantity
0,100559pat0001pr0002,2023,3
1,100559pat0001pr0002,2024,1
2,100559pat0001pr0002,2025,3
3,11126,2013,25
4,11126,2014,13
...,...,...,...
6448,flex08c12,1995,1
6449,flex08c12,1997,4
6450,flex08c12,2003,2
6451,upn0200,1998,1


In [47]:
parent_child_parts = duckdb.sql(
"""
SELECT 
    min_parent_year,
    max_parent_year,
    parent_part_num,
    child_part_num,
    min_child_year,
    max_child_year
FROM joined
"""
).df() #category and part names can be re-joined in Power BI to create a more efficient workflow. 

parent_child_parts

,min_parent_year,max_parent_year,parent_part_num,child_part_num,min_child_year,max_child_year
0,2015,2021,20512pr0001,14518,2013,2021
1,1991,2013,2547,87587,2013,2013
2,2003,2005,40373,40375,2001,2005
3,2003,2005,40374,40375,2001,2005
4,2001,2005,40375,43890c01,2003,2003
...,...,...,...,...,...,...
480,1983,2006,4315,4474,1985,1991
481,1983,2006,4315,30083,2001,2003
482,1994,2003,4625,4474,1985,1991
483,1994,2003,4625,30083,2001,2003


In [54]:
parts_category = duckdb.sql("SELECT p.part_num, p.name, pc.name AS category FROM parts p JOIN part_categories pc ON p.part_cat_id = pc.id WHERE p.part_num IN(SELECT part_num FROM part_qty_per_year)").df()
parts_category

,part_num,name,category
0,100559pat0001pr0002,"Animal, Dog, Dachshund with Vibrant Yellow Har...",Animals / Creatures
1,11126,Rip Cord Flexible with Handle,Tools
2,11208,"Wheel 14mm D. x 9.9mm with Centre Groove, Fake...",Wheels and Tyres
3,11213,Plate Round 6 x 6 with Hole,Plates Round Curved and Dishes
4,11299,Ladder 16 x 3.5 with Side Supports,"Bars, Ladders and Fences"
...,...,...,...
460,98459,"Duplo Door / Lid, Wood Effect","Duplo, Quatro and Primo"
461,98562,Large Figure Weapon Claw / Handcuff,Large Buildable Figures
462,98563,"Large Figure Weapon, Zamor Sphere Launcher, To...",Large Buildable Figures
463,flex08c12,Technic Flex Cable 12L,Technic Special


Pieces of information that will help determine a conclusion:

- did the parent part retire before the child part was introduced?
    - if so, what was the period between one part being retired and the ther being introduced?
    - if not, what was the period of time where both parts were in production?
        - if both are retired, did they retire at the same time?
        - are both parts still in production?

- any instances where the parent part was in production longer than the child part?
    - may indicate the 'failure' of a child part...
    - but may be anomalous if the part is hyper-specifc to one set or scenario, or if the part is relatively new.


- in years where a parent part and a child part coexisted, which was used more in sets? (using the part_qty_per_year table)
    - bar-chart timelines, per category, of parent part quantity vs child part (POWER BI)

- 're-introduction' of a part that may have been perceived as discontinued. Set an arbitrary time gap, eg. 5 years.







